In [6]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

In [7]:
df = pd.read_csv("../data/week1_features.csv")
df.columns = df.columns.str.replace('[', '_', regex=False).str.replace(']', '_', regex=False).str.replace(' ', '_', regex=False)

feature_cols = [col for col in df.columns if 'roll' in col]
X = df[feature_cols]
y = df["Machine_failure"]
print("Class distribution:", y.value_counts().to_dict())

Class distribution: {0: 9661, 1: 339}


In [8]:
skf = StratifiedKFold(n_splits=5)
baseline_f1 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model = lgb.LGBMClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    baseline_f1.append(f1)

print(f"Baseline Avg Macro F1 (No SMOTE): {sum(baseline_f1)/len(baseline_f1):.4f}")

[LightGBM] [Info] Number of positive: 272, number of negative: 7728
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11101
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034000 -> initscore=-3.346803
[LightGBM] [Info] Start training from score -3.346803
[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002104 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11218
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.033875 -> initscore=-3.350616
[LightGBM] [Info] Start training from score -3.350616
[LightGBM] [Info] 

In [9]:
smote = SMOTE(random_state=42)
smote_f1 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    
    model = lgb.LGBMClassifier(n_estimators=100, random_state=42)
    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    smote_f1.append(f1)

print(f"SMOTE Avg Macro F1: {sum(smote_f1)/len(smote_f1):.4f}")

[LightGBM] [Info] Number of positive: 7728, number of negative: 7728
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004679 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12739
[LightGBM] [Info] Number of data points in the train set: 15456, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 7729, number of negative: 7729
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004410 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12740
[LightGBM] [Info] Number of data points in the train set: 15458, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 7729, number of negative: 7729
[LightGBM] [Info] Auto-choosing col-wise multi-threadin

In [10]:
print("=== Model Comparison ===")
print(f"Baseline Macro F1: {sum(baseline_f1)/len(baseline_f1):.4f}")
print(f"SMOTE Macro F1:    {sum(smote_f1)/len(smote_f1):.4f}")
improvement = sum(smote_f1)/len(smote_f1) - sum(baseline_f1)/len(baseline_f1)
print(f"Improvement: {improvement:.4f}")

=== Model Comparison ===
Baseline Macro F1: 0.4655
SMOTE Macro F1:    0.4797
Improvement: 0.0142
